In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

from tqdm import tqdm
import json
import torch

import faiss
from uuid import uuid4

from langchain_community.vectorstores import Chroma, FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain.docstore.document import Document

## Overview

In [10]:
workspace = '/project/lt200304-dipmt/paweekorn'
train_df = pd.read_csv(f"{workspace}/data/train_set.csv")

with open(f'{workspace}/data/WIPO.json', 'r') as f:
    wipo = json.load(f)
    wipo = {int(k): v for k, v in wipo.items()}


train_df = train_df.drop_duplicates(['ENG']).reset_index(drop=True)
print(train_df.shape)
train_df['WIPO'] = train_df['NAME'].map(wipo)
train_df.head()

(37357, 3)


,NAME,ENG,THA,WIPO
0,25,"Clothing, namely, blousons, dress shirts, shir...",เครื่องแต่งกาย ได้แก่ เสื้อแจ๊คเก็ตสั้นถึงเอว ...,"Clothing, footwear, headwear."
1,25,"headgear, namely, bonnets, headscarves, beanie...",เครื่องคลุมศีรษะ ได้แก่ หมวกปีกกว้างสตรีที่มีส...,"Clothing, footwear, headwear."
2,25,"waterproof clothing, namely, vests",ชุดกันน้ำ ได้แก่ เสื้อกั๊ก,"Clothing, footwear, headwear."
3,25,veils being clothing,ผ้าคลุมที่เป็นเครื่องแต่งกาย,"Clothing, footwear, headwear."
4,25,Valenki being felted boots,รองเท้ากันหนาวที่เป็นรองเท้าบูทขนสัตว์,"Clothing, footwear, headwear."


In [11]:
documents = []
for _, row in tqdm(train_df.iterrows()):
    content = row['ENG']
    meta = {'wipo': row['WIPO'], 'thai': row['THA']}
    documents.append(Document(page_content=content, metadata=meta))

uuids = [str(uuid4()) for _ in range(len(documents))]
documents[0]

37357it [00:01, 26932.93it/s]


Document(metadata={'wipo': 'Clothing, footwear, headwear.', 'thai': 'เครื่องแต่งกาย ได้แก่ เสื้อแจ๊คเก็ตสั้นถึงเอว เสื้อเชิ้ตสำหรับใส่ออกงาน เสื้อเชิ้ต เสื้อโค๊ท กางเกงขายาว เสื้อกันหนาวที่สวมทางศีรษะ เสื้อแจ๊คเก็ตคาร์ดิแกน เสื้อกั๊ก เสื้อกันหนาวที่สวมทางศีรษะ กางเกงขายาว กางเกงวอร์ม กางเกงขาสั้นเหนือเข่า กางเกงยีนส์ เสื้อยืด ชุดสูท เสื้อโอเวอร์โค๊ท เสื้อแจ๊คเก็ตที่มีหมวกคลุม เสื้อกันฝน เสื้อแขนยาวอย่างหนาใช้สำหรับออกกำลังกาย เนคไท เสื้อคลุมยาวของบาทหลวง'}, page_content='Clothing, namely, blousons, dress shirts, shirts, coats, pants, pullovers, cardigans, waistcoats, jumpers, pants, sweat pants, bermuda shorts, jeans, T-shirts, suits, overcoats, anoraks, raincoats, sweatshirts, ties, albs')

## RAG

In [13]:
embedding_model = HuggingFaceEmbeddings(model_name=f"{workspace}/models/bge-m3")
d_model = len(embedding_model.embed_query('Hello World'))
d_model

1024

In [14]:
res = faiss.StandardGpuResources()

index_flat = faiss.IndexFlatL2(d_model)
gpu_index_flat = faiss.index_cpu_to_gpu(res, 0, index_flat)  # make it into a gpu index

# make it into a gpu index
vector_store = FAISS(
    embedding_function=embedding_model,
    index=gpu_index_flat,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)
vector_store.add_documents(documents=documents, ids=uuids)

['4eb61878-152a-44d2-99df-4cef01a64a22',
 'd847387a-65b5-4670-b4aa-7c32132b6a66',
 'd784bf4d-ffc1-4289-878a-cdff1c31464b',
 '1df3020d-156f-4665-8225-1809e7c3147e',
 '0121127f-64e4-4ee8-9576-50064243892c',
 'c31ca5e6-787e-4f0f-a139-5fe61dc630da',
 'e667497d-ee9c-4c8b-a311-82dcf808b3b1',
 'ccb1bbfc-e07b-45bb-b7c7-d3e1bcae62c2',
 'fa560653-467f-4b9e-ad90-40f18d6f0b25',
 '1faf5015-a495-4260-bae3-fed9ebe18c6e',
 '15b73da3-80d2-444a-9f88-06edce6fc800',
 'd51a5b75-5201-4406-9070-a7231f48a63a',
 '2d326bd4-b9a8-4105-9154-f6a1cdad5587',
 'c3fca9e2-746c-4f8d-9a92-bfd7282c6593',
 '7cb17bb5-d3e1-491f-ab04-7988bbf925c1',
 'd81c97cd-22eb-47e5-9d29-8c0a014bc45e',
 '360542e2-9adf-4924-a64b-0a4a76df127e',
 '00f4e484-7296-4e74-b7dc-d6752cb6a19c',
 '18c5dbc2-50f0-42ab-8012-13a40ce409fd',
 '9e33c30e-8c24-4418-8960-0210fd0d54fc',
 'a302a6a7-2315-4507-9865-3074728e1f36',
 '74d98396-ccff-4980-8629-28bbce66f43a',
 '39e40c74-a22d-4439-aeed-e6947b6dbe46',
 'ee79ac1b-877c-44d8-889a-78ef5bb2f870',
 '06217bc8-0a0a-

In [15]:
os.makedirs(f'{workspace}/faiss_train', exist_ok=True)

# Move the index to CPU before saving
cpu_index = faiss.index_gpu_to_cpu(vector_store.index)
faiss.write_index(cpu_index, f"{workspace}/faiss_train/index.faiss")

# Re-create the FAISS vector store from the saved index, docstore, and index_to_docstore_id
vector_store_loaded = FAISS(
    embedding_function=embedding_model,
    index=cpu_index,
    docstore=vector_store.docstore,
    index_to_docstore_id=vector_store.index_to_docstore_id,
)

# Now you can save the other components using save_local
vector_store_loaded.save_local(f"{workspace}/faiss_train")

## Demo

In [22]:
def get_relevant_docs(query, k=4):
    docs = vector_store.similarity_search(query, k=k)

    relevant = ""
    for i, doc in enumerate(docs[1:]):
        relevant += f'''**Example {i+1}**
Source: {doc.page_content}
Translation: {doc.metadata['thai']}
\n'''

    return relevant

sample = train_df.loc[3]
print(f"Source: {sample['ENG']}")
print(f"Translation: {sample['THA']}\n")

print("## Retrieval Result")
print(get_relevant_docs(sample['ENG']))

Source: veils being clothing
Translation: ผ้าคลุมที่เป็นเครื่องแต่งกาย

## Retrieval Result
**Example 1**
Source: belts being clothing
Translation: เข็มขัดใช้เป็นเครื่องแต่งกาย

**Example 2**
Source: layettes being clothing
Translation: ชุดสำหรับเด็กแรกเกิด

**Example 3**
Source: trunks being clothing
Translation: กางเกงในชายขาสั้นรัดรูปที่เป็นเครื่องแต่งกาย


